In [1]:
%matplotlib inline

In [2]:
#Import your libraries here

import csv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
from pathlib import Path
from datetime import datetime

In [3]:
#Import your modules here
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from utils.ad_parsing import ad_parsing_utils as adp
from utils.fetch_data import (
                                ScrapeSelection,
                                collect_listing_urls_for_routes_parallel_streaming,
                                download_and_parse_listing_batch_streaming, 
                                load_valid_deal_types,
                                load_valid_geo_paths,
                                load_valid_property_types
                                )

print(PROJECT_ROOT)

D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_1


In [4]:
RAW_HTML_DIR = PROJECT_ROOT / "data" / "raw_listing_html"
RAW_HTML_DIR.mkdir(parents=True, exist_ok=True)

for file_path in RAW_HTML_DIR.glob("*.html"):
    file_path.unlink()

TAXONOMY_DIR = PROJECT_ROOT / "data" / "taxonomy"
DATA_DIR = PROJECT_ROOT / "data"

print(TAXONOMY_DIR)

# ============================================================
# Run mode
# ============================================================

RUN_MODE = "resume"
# Options:
# "new"    = create a fresh independent run
# "resume" = continue a previous interrupted run


# Use only when RUN_MODE = "resume"
RESUME_RUN_ID = "20260608_081646"  # replace with the actual old RUN_ID


if RUN_MODE == "new":
    RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
elif RUN_MODE == "resume":
    RUN_ID = RESUME_RUN_ID
else:
    raise ValueError("RUN_MODE must be either 'new' or 'resume'")

ROUTE_DISCOVERY_DIR = (
    PROJECT_ROOT
    / "data"
    / "route_discovery_runs"
    / f"sales_full_{RUN_ID}"
)

PARSED_OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "parsed_sales_runs"
    / f"parsed_sales_full_{RUN_ID}"
)

if RUN_MODE == "new":
    ROUTE_DISCOVERY_DIR.mkdir(parents=True, exist_ok=False)
    PARSED_OUTPUT_DIR.mkdir(parents=True, exist_ok=False)

elif RUN_MODE == "resume":
    if not ROUTE_DISCOVERY_DIR.exists():
        raise FileNotFoundError(f"Cannot resume. Missing route folder: {ROUTE_DISCOVERY_DIR}")

    PARSED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("RUN_MODE:", RUN_MODE)
print("RUN_ID:", RUN_ID)
print("Route discovery dir:", ROUTE_DISCOVERY_DIR)
print("Parsed output dir:", PARSED_OUTPUT_DIR)

D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_1\data\taxonomy
RUN_MODE: resume
RUN_ID: 20260608_081646
Route discovery dir: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_1\data\route_discovery_runs\sales_full_20260608_081646
Parsed output dir: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_1\data\parsed_sales_runs\parsed_sales_full_20260608_081646


## Call the generated url and obtain a list of urls with ads to the related query

In [5]:
valid_deal_types = load_valid_deal_types(TAXONOMY_DIR / "valid_deal_types.csv")
valid_geo_paths = load_valid_geo_paths(TAXONOMY_DIR / "valid_geo_paths.csv")
valid_property_types = load_valid_property_types(TAXONOMY_DIR / "valid_property_types.csv")


all_sales_geo_paths = [
    geo_path
    for deal_type, geo_path in valid_geo_paths
    if deal_type == "prodazhbi"
]

selection = ScrapeSelection(
    deal_types=["prodazhbi"],
    geo_paths=all_sales_geo_paths,
    property_types=list(valid_property_types),
)

route_urls = selection.build_urls(
    valid_deal_types,
    valid_geo_paths,
    valid_property_types,
)

print("Geo paths:", len(all_sales_geo_paths))
print("Property types:", len(valid_property_types))
print("Route URLs:", len(route_urls))
print(route_urls[:10])

run_summary = collect_listing_urls_for_routes_parallel_streaming(
    route_urls=route_urls,
    max_workers=24,
    max_pages=200,
    delay_seconds=0.6,
    checkpoint_dir=ROUTE_DISCOVERY_DIR,
    progress_every=100,
)

run_summary

Geo paths: 4375
Property types: 22
Route URLs: 96250
['https://www.imot.bg/obiavi/prodazhbi/oblast-burgas/s-malina/mezonet', 'https://www.imot.bg/obiavi/prodazhbi/oblast-burgas/s-malina/mnogostaen', 'https://www.imot.bg/obiavi/prodazhbi/oblast-burgas/s-malina/kashta', 'https://www.imot.bg/obiavi/prodazhbi/oblast-burgas/s-malina/chetiristaen', 'https://www.imot.bg/obiavi/prodazhbi/oblast-burgas/s-malina/atelie-tavan', 'https://www.imot.bg/obiavi/prodazhbi/oblast-burgas/s-malina/vila', 'https://www.imot.bg/obiavi/prodazhbi/oblast-burgas/s-malina/myasto', 'https://www.imot.bg/obiavi/prodazhbi/oblast-burgas/s-malina/dvustaen', 'https://www.imot.bg/obiavi/prodazhbi/oblast-burgas/s-malina/magazin', 'https://www.imot.bg/obiavi/prodazhbi/oblast-burgas/s-malina/garazh-parkomyasto']
Total routes: 96250
Already completed: 72737
Pending routes: 23513
[72738/96250] routes completed | new this run=1 | route listings=0 | pages=1 | stop=no_listing_urls | url=https://www.imot.bg/obiavi/prodazhbi/oblast

{'total_routes': 96250,
 'already_completed': 72737,
 'newly_completed': 23513,
 'new_page_rows': 29040,
 'new_listing_rows': 82835,
 'route_results_path': WindowsPath('D:/users/kamen.dimitrov/desktop/SOFTUNI/AI_and_ML_upskill_program/machine_learning/08_final_project_1/data/route_discovery_runs/sales_full_20260608_081646/route_results.csv'),
 'page_results_path': WindowsPath('D:/users/kamen.dimitrov/desktop/SOFTUNI/AI_and_ML_upskill_program/machine_learning/08_final_project_1/data/route_discovery_runs/sales_full_20260608_081646/page_results.csv'),
 'listing_urls_path': WindowsPath('D:/users/kamen.dimitrov/desktop/SOFTUNI/AI_and_ML_upskill_program/machine_learning/08_final_project_1/data/route_discovery_runs/sales_full_20260608_081646/listing_urls_raw.csv')}

In [5]:
listing_urls_raw_path = ROUTE_DISCOVERY_DIR / "listing_urls_raw.csv"
listing_urls_final_path = ROUTE_DISCOVERY_DIR / "listing_urls_unique.csv"

listing_urls_df = pd.read_csv(listing_urls_raw_path)

unique_listing_urls_df = (
    listing_urls_df[["listing_url"]]
    .drop_duplicates()
    .sort_values("listing_url")
    .reset_index(drop=True)
)

unique_listing_urls_df.to_csv(listing_urls_final_path, index=False)

print("Run ID:", RUN_ID)
print("Raw listing rows:", len(listing_urls_df))
print("Unique listing URLs:", len(unique_listing_urls_df))
print("Saved to:", listing_urls_final_path)

Run ID: 20260608_081646
Raw listing rows: 274173
Unique listing URLs: 168076
Saved to: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_1\data\route_discovery_runs\sales_full_20260608_081646\listing_urls_unique.csv


## Read the list of URLS from the generated query

In [8]:
# ============================================================
# Load discovered listing URLs from current route crawl run
# Then download + parse listings using streaming pipeline
# ============================================================

# Use the ROUTE_DISCOVERY_DIR already created in Cell 3.
# Do NOT redefine it here.

RAW_LISTING_URLS_CSV = ROUTE_DISCOVERY_DIR / "listing_urls_unique.csv"

PARSED_OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "parsed_sales_runs"
    / f"parsed_sales_full_{RUN_ID}"
)

PARSED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 1. Load listing URLs from route crawl output
# ------------------------------------------------------------

df_urls = pd.read_csv(RAW_LISTING_URLS_CSV)

listing_urls = (
    df_urls["listing_url"]
    .dropna()
    .drop_duplicates()
    .tolist()
)

print("RUN_ID:", RUN_ID)
print("Route discovery dir:", ROUTE_DISCOVERY_DIR)
print("Parsed output dir:", PARSED_OUTPUT_DIR)
print("Loaded unique listing URLs:", len(listing_urls))
print(listing_urls[:5])


# ------------------------------------------------------------
# 2. Download + parse listings using streaming/resumable method
# ------------------------------------------------------------

download_parse_summary = download_and_parse_listing_batch_streaming(
    listing_urls=listing_urls,
    output_dir=PARSED_OUTPUT_DIR,
    max_workers=24,
    limit=None,          # None = process all
    chunk_size=3000,     # keeps futures bounded
    progress_every=300,
)

download_parse_summary


# ------------------------------------------------------------
# 3. Optional: inspect parsed output
# ------------------------------------------------------------

parsed_csv = PARSED_OUTPUT_DIR / "parsed_listings.csv"
manifest_csv = PARSED_OUTPUT_DIR / "download_manifest.csv"

print("Download manifest:", manifest_csv)
print("Parsed listings:", parsed_csv)

if parsed_csv.exists():
    df_parsed = pd.read_csv(parsed_csv)
    print("Parsed rows:", len(df_parsed))
    display(df_parsed.head())
else:
    print("No parsed listings CSV created yet.")

RUN_ID: 20260608_081646
Route discovery dir: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_1\data\route_discovery_runs\sales_full_20260608_081646
Parsed output dir: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_1\data\parsed_sales_runs\parsed_sales_full_20260608_081646
Loaded unique listing URLs: 168076
['https://www.imot.bg/obiava-1a119460323900784-prodava-ednostaen-apartament-grad-sofiya-mladost-1', 'https://www.imot.bg/obiava-1a119988255316983-prodava-ednostaen-apartament-grad-sofiya-gorna-banya', 'https://www.imot.bg/obiava-1a126881929164804-prodava-ednostaen-apartament-grad-pleven-druzhba-1-bivshiyat-bikarnik', 'https://www.imot.bg/obiava-1a127539284802892-prodava-ednostaen-apartament-oblast-dobrich-gr-balchik', 'https://www.imot.bg/obiava-1a129794411209548-prodava-ednostaen-apartament-oblast-blagoevgrad-gr-bansko']
Total listing URLs: 168076
Already completed: 168076
Pendin

C:\Users\kamen.dimitrov\AppData\Local\Temp\ipykernel_42092\1664623396.py:68: DtypeWarning: Columns (25) have mixed types. Specify dtype option on import or set low_memory=False.
  df_parsed = pd.read_csv(parsed_csv)


Parsed rows: 164982


,ad_id,ad_url,ad_url_path,ad_url_slug,html_path,listing_type,title,deal_raw,property_type_raw,title_city_raw,...,location_raw,description_raw,description_clean,features_raw,features_pipe,features_count,published_raw,views,training_eligible,parse_error
0,1a155107752004273,https://www.imot.bg/obiava-1a155107752004273-p...,/obiava-1a155107752004273-prodava-ednostaen-ap...,obiava-1a155107752004273-prodava-ednostaen-apa...,data\raw_listing_html\948e49be77a55a7a.html,single_property_listing,"Продава 1-СТАЕН в област Пазарджик, гр. Велинг...",Продава,1-СТАЕН,област Пазарджик,...,"област Пазарджик, гр. Велинград, Цар Самуил",ЕДНОСТАЕН АПАРТАМЕНТ В ЦЕНТЪРА НА ВЕЛИНГРАД ...,ЕДНОСТАЕН АПАРТАМЕНТ В ЦЕНТЪРА НА ВЕЛИНГРАД ...,Тухла\nАсансьор\nС гараж\nС паркинг\nЛизинг,Тухла|Асансьор|С гараж|С паркинг|Лизинг,5,NaN,9.0,True,NaN
1,1a146979303032401,https://www.imot.bg/obiava-1a146979303032401-p...,/obiava-1a146979303032401-prodava-ednostaen-ap...,obiava-1a146979303032401-prodava-ednostaen-apa...,data\raw_listing_html\9b4948bfbf128a8a.html,single_property_listing,"Продава 1-СТАЕН в град София, Малинова долина ...",Продава,1-СТАЕН,град София,...,"град София, Малинова долина",🏙️ ЕДНОСТАЕН АПАРТАМЕНТ В НОВА СГРАДА НА ЕТАП ...,🏙️ ЕДНОСТАЕН АПАРТАМЕНТ В НОВА СГРАДА НА ЕТАП ...,Тухла,Тухла,1,NaN,734.0,True,NaN
2,1a140680337846667,https://www.imot.bg/obiava-1a140680337846667-p...,/obiava-1a140680337846667-prodava-ednostaen-ap...,obiava-1a140680337846667-prodava-ednostaen-apa...,data\raw_listing_html\d014b1815f2f9c9b.html,single_property_listing,"Продава 1-СТАЕН в град Варна, м-т Ален мак - 3...",Продава,1-СТАЕН,град Варна,...,"град Варна, м-т Ален мак, спортпалас","апартамент в затворен комплекс,до комплекс спо...","апартамент в затворен комплекс,до комплекс спо...",Тухла\nЛизинг\nБартер,Тухла|Лизинг|Бартер,3,"Публикувана в 13:42 на 31 юли, 2014 год.",13987.0,True,NaN
3,1a154997703660554,https://www.imot.bg/obiava-1a154997703660554-p...,/obiava-1a154997703660554-prodava-ednostaen-ap...,obiava-1a154997703660554-prodava-ednostaen-apa...,data\raw_listing_html\6a625253f270be91.html,single_property_listing,"Продава 1-СТАЕН в област Бургас, гр. Свети Вла...",Продава,1-СТАЕН,област Бургас,...,"област Бургас, гр. Свети Влас",Собственик продава студио в жилищен комплекс И...,Собственик продава студио в жилищен комплекс И...,Тухла\nАсансьор\nС паркинг\nЛизинг\nИнтернет в...,Тухла|Асансьор|С паркинг|Лизинг|Интернет връзк...,11,NaN,24930.0,True,NaN
4,1a155107630313014,https://www.imot.bg/obiava-1a155107630313014-p...,/obiava-1a155107630313014-prodava-ednostaen-ap...,obiava-1a155107630313014-prodava-ednostaen-apa...,data\raw_listing_html\a31953e77840ec11.html,single_property_listing,"Продава 1-СТАЕН в област Пазарджик, гр. Велинг...",Продава,1-СТАЕН,област Пазарджик,...,"област Пазарджик, гр. Велинград, Кисловодск",НОВО СТРОИТЕЛСТВО -гарантиран строител\n🏡 Пано...,НОВО СТРОИТЕЛСТВО -гарантиран строител 🏡 Панор...,Тухла\nАсансьор\nС гараж\nС паркинг\nЛизинг\nИ...,Тухла|Асансьор|С гараж|С паркинг|Лизинг|Интерн...,7,NaN,4.0,True,NaN


In [10]:
csv_path = PARSED_OUTPUT_DIR / "parsed_listings.csv"
parquet_path = PARSED_OUTPUT_DIR / "parsed_listings.parquet"

df = pd.read_csv(csv_path)

df.to_parquet(
    parquet_path,
    index=False,
    engine="pyarrow",
    compression="snappy",
)

print("Saved:", parquet_path)
print("Rows:", len(df))
print("Columns:", len(df.columns))

C:\Users\kamen.dimitrov\AppData\Local\Temp\ipykernel_42092\773456746.py:4: DtypeWarning: Columns (25) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path)


Saved: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_1\data\parsed_sales_runs\parsed_sales_full_20260608_081646\parsed_listings.parquet
Rows: 164982
Columns: 39
